# Fine-tuning Whisper (small) pour le fulfulde (fuv) — Adamaoua, Cameroun

Ce notebook :
1. Monte ton Google Drive
2. Clone/utilise le code du repo `fuv-stt-whisper`
3. Installe les dependances
4. Verifie ton dataset (audio + `metadata.json`)
5. Lance le fine-tuning de `openai/whisper-small`
6. Teste le modele obtenu sur un nouvel audio

Avant de commencer : pousse ton dossier de dataset sur ton Drive avec cette structure :

```
MyDrive/fuv-stt-dataset/
├── audio/
│   ├── 0001.wav
│   ├── 0002.wav
│   └── ...
└── metadata.json
```

Voir le README du repo pour le format exact de `metadata.json`.

## 1. Monter Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Récupérer le code du projet\n\nRemplace l'URL par celle de ton repo GitHub une fois que tu l'auras créé et poussé.

In [ ]:
%cd /content
!git clone https://github.com/bonopassale/fuv-stt-whisper.git
%cd fuv-stt-whisper

## 3. Installer les dépendances

In [ ]:
!pip install -r requirements.txt --quiet

## 4. Vérifier / ajuster la configuration\n\nOuvre `src/config.py` si besoin (chemin Drive, taille du modele, hyperparametres). Par defaut tout pointe vers `/content/drive/MyDrive/fuv-stt-dataset`.

In [ ]:
!cat src/config.py

## 5. Vérification rapide du dataset (avant d'entraîner)\n\nCharge juste les métadonnées et vérifie que tous les fichiers audio existent, sans lancer l'entraînement complet.

In [ ]:
%cd src
from dataset import load_raw_dataset
raw = load_raw_dataset()
print(raw)
print("\nExemple :", raw["train"][0]["text"])

## 6. Lancer le fine-tuning\n\nSelon la taille de ton dataset (~2h d'audio) et le GPU Colab attribué, compte entre 30 minutes et 2 heures pour `max_steps=1000`.

In [ ]:
!python train.py

## 7. Tester le modèle fine-tuné\n\nDépose un fichier audio de test (fulfulde, différent du dataset d'entraînement) et transcris-le.

In [ ]:
!python inference.py --audio /content/drive/MyDrive/fuv-stt-dataset/test_audio.wav

## 8. (Optionnel) Publier sur Hugging Face Hub\n\nPasse `push_to_hub = True` dans `config.py` avant l'entraînement, ou pousse manuellement le dossier `output_dir` a posteriori :

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

# Puis, si push_to_hub=False pendant l'entrainement :
# from transformers import WhisperForConditionalGeneration, WhisperProcessor
# from config import train_config
# model = WhisperForConditionalGeneration.from_pretrained(train_config.output_dir)
# processor = WhisperProcessor.from_pretrained(train_config.output_dir)
# model.push_to_hub("bonopassale/whisper-small-fuv")
# processor.push_to_hub("bonopassale/whisper-small-fuv")